# k-means (Lloyd) — Least squares quantization in PCM (1982)

_Papers · Classical ML_

**Alternate two simple steps — snap each point to its nearest center, move each center to the mean of its points — and a clustering's total squared error can only fall.**

---

This notebook is a practice scaffold for the **k-means (Lloyd) — Least squares quantization in PCM (1982)** lesson. We build Lloyd's algorithm one step at a time, watch the distortion fall monotonically, and check it against scikit-learn. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

### Step 1 — Write Lloyd's algorithm from scratch

Lloyd's algorithm alternates two moves until the centers stop moving. **Assign:** give every point to its nearest center (the nearest-neighbor condition). **Update:** move each center to the mean of the points assigned to it (the centroid condition). Both moves can only lower the within-cluster squared error — the *distortion* J — so the history we record below is monotone non-increasing.

In [ ]:
from sklearn.cluster import KMeans
from itertools import permutations

def my_kmeans(X, k, init, max_iter=100, tol=1e-12):
    """Lloyd's algorithm from scratch. Returns final centers, labels, distortion history."""
    C = np.asarray(init, dtype=float).copy()
    history = []
    for _ in range(max_iter):
        # ASSIGN: squared Euclidean distance from every point to every center -> nearest
        D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)    # shape (n, k)
        labels = D.argmin(axis=1)                                 # nearest-neighbor condition
        distortion = float(D[np.arange(len(X)), labels].sum())    # distortion J (inertia)
        history.append(distortion)
        # UPDATE: each center -> mean of its assigned points (centroid condition)
        newC = np.array([X[labels == j].mean(axis=0) if np.any(labels == j) else C[j]
                         for j in range(k)])
        if np.allclose(newC, C, atol=tol):                        # converged: centers stopped moving
            C = newC
            break
        C = newC
    # final distortion at the converged centers
    D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
    labels = D.argmin(axis=1)
    final_distortion = float(D[np.arange(len(X)), labels].sum())
    history.append(final_distortion)
    return C, labels, history

### Step 2 — Run it on three blobs from a deliberately poor start

We make three clearly separated 2-D blobs but start all three centers bunched in one corner. That bad start forces several visible monotone steps before convergence. Printing the distortion history lets us confirm it never increases.

In [ ]:
# ---- toy 2-D set: three clear blobs ----
rng = np.random.default_rng(0)
centers_true = np.array([[0, 0], [6, 6], [0, 7]], float)
X = np.vstack([c + rng.normal(0, 0.8, (20, 2)) for c in centers_true])
k = 3

# deliberately POOR start (all near one corner) so we see several monotone steps
init = np.array([[0., 0.], [0.5, 0.5], [1.0, 0.0]])
C, labels, hist = my_kmeans(X, k, init)

print("distortion each iteration:", [round(h, 4) for h in hist])    # falls every step
monotone = all(hist[i] >= hist[i+1] - 1e-9 for i in range(len(hist)-1))
print("monotone non-increasing:", monotone)                         # True
print("our final inertia:", round(hist[-1], 4))                     # 66.2327

### Step 3 — Check against scikit-learn (the oracle)

From the *same* initialization, `sklearn.cluster.KMeans` runs the identical Lloyd iteration, so its final inertia must match ours. We also confirm the cluster assignments agree up to a relabeling (cluster identities are arbitrary, only the partition matters).

In [ ]:
# ---- THE ORACLE: same start -> must equal sklearn.cluster.KMeans ----
km = KMeans(n_clusters=k, init=init, n_init=1, max_iter=100, tol=1e-12).fit(X)
print("sklearn inertia:", round(km.inertia_, 4))                     # 66.2327
print("inertia allclose:", np.allclose(hist[-1], km.inertia_, atol=1e-4))   # True

def same_partition(a, b, k):                  # labels match up to a renaming?
    return any(np.array_equal(np.array(p)[a], b) for p in permutations(range(k)))

print("labels match (up to perm):", same_partition(labels, km.labels_, k))  # True

### Step 4 — Replay the one-iteration worked example by hand

Finally we reproduce the lesson's small worked example: 6 points, k=2, centers parked in the corners. We do one assign step (and read off the distortion), then one update step (centers move to the means), and watch the distortion drop sharply — the algorithm in miniature.

In [ ]:
# ---- recompute the one-iteration worked example: 6 pts, k=2, corner centers ----
Xe = np.array([[1, 1], [1, 3], [2, 2], [8, 8], [9, 7], [7, 9]], float)
ce = np.array([[0, 0], [10, 10]], float)

# one ASSIGN step
D = ((Xe[:, None, :] - ce[None, :, :]) ** 2).sum(2)
lab = D.argmin(1)
distortion_after_assign = round(D[np.arange(6), lab].sum(), 4)
print("worked: assignments", lab.tolist(),
      "| distortion after assign", distortion_after_assign)         # 48.0

# one UPDATE step
new = np.array([Xe[lab == j].mean(0) for j in range(2)])
D2 = ((Xe[:, None, :] - new[None, :, :]) ** 2).sum(2)
distortion_after_update = round(D2[np.arange(6), D2.argmin(1)].sum(), 4)
print("worked: new centers", np.round(new, 3).tolist(),
      "| distortion after update", distortion_after_update)         # 6.6667

## Visualize the data & results

_From a deliberately poor start, does our from-scratch Lloyd loop drive the within-cluster distortion down monotonically until it flattens — and does the converged value match sklearn.cluster.KMeans exactly?_

### Step 1 — Rebuild the blobs and a compact Lloyd loop

We regenerate the same three blobs and define a trimmed `lloyd` helper that returns just the distortion history. Running it from the poor start and comparing to scikit-learn confirms our curve ends exactly where the oracle does.

In [ ]:
from sklearn.cluster import KMeans
rng = np.random.default_rng(0)

# three toy blobs, 60 points total
centers_true = np.array([[0, 0], [6, 6], [0, 7]], float)
X = np.vstack([c + rng.normal(0, 0.8, (20, 2)) for c in centers_true])
k = 3

def lloyd(X, k, init, iters=15):
    C = np.asarray(init, float).copy()
    hist = []
    for _ in range(iters):
        D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(2)
        lab = D.argmin(1)
        hist.append(float(D[np.arange(len(X)), lab].sum()))
        newC = np.array([X[lab == j].mean(0) if np.any(lab == j) else C[j] for j in range(k)])
        if np.allclose(newC, C):
            break
        C = newC
    D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(2)
    hist.append(float(D[np.arange(len(X)), D.argmin(1)].sum()))
    return hist

init = np.array([[0., 0.], [0.5, 0.5], [1.0, 0.0]])   # poor start
hist = lloyd(X, k, init)
km = KMeans(n_clusters=k, init=init, n_init=1, max_iter=100, tol=1e-12).fit(X)
print("our distortion curve:", [round(h, 2) for h in hist])   # [2257.44,267.42,67.57,66.23,66.23]
print("our final:", round(hist[-1], 4), " sklearn:", round(km.inertia_, 4))   # 66.2327 == 66.2327

### Step 2 — Ablation: k-means++ vs random initialization

Lloyd only finds a *local* optimum, so the starting centers matter a lot. We compare two initializations over 50 seeds: plain random points versus the k-means++ scheme (which spreads the first centers out by sampling proportional to squared distance). Random starts scatter into much worse optima, while k-means++ reliably lands on the good clustering.

In [ ]:
# ablation: k-means++ vs random initialization over 50 seeds
def kmpp(X, k, r):
    idx = [r.integers(len(X))]
    for _ in range(k - 1):
        d = np.min(((X[:, None, :] - X[idx][None, :, :]) ** 2).sum(2), axis=1)
        idx.append(r.choice(len(X), p=d / d.sum()))
    return X[idx].copy()

def trial(kind):
    fins = []
    for s in range(50):
        r = np.random.default_rng(100 + s)
        if kind == "rand":
            ini = X[r.choice(len(X), k, replace=False)]
        else:
            ini = kmpp(X, k, r)
        fins.append(lloyd(X, k, ini)[-1])
    return np.array(fins)

rand = trial("rand")
pp = trial("kmpp")
print("random init  mean %.2f  worst %.2f" % (rand.mean(), rand.max()))   # ~209, ~547
print("k-means++    mean %.2f  worst %.2f" % (pp.mean(), pp.max()))       # 66.23, 66.23

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Verify the centroid condition by hand: which single point $\mu$ minimizes $\sum_{x\in S}\lVert x-\mu\rVert^2$ for $S=\{(1,1),(1,3),(2,2)\}$, and what is the minimum value?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Take the gradient: $\sum_{x}-2(x-\mu)=0 \Rightarrow \mu=\text{mean}(S)$. — _The cost is a convex bowl in $\mu$; its minimum is where the slope is zero._
- Mean $=\big(\tfrac{1+1+2}{3},\tfrac{1+3+2}{3}\big)=(1.333,2.0)$. — _Coordinate-wise average._
- Squared distances to $\mu$: $(1,1)\!:\,0.111+1=1.111$; $(1,3)\!:\,0.111+1=1.111$; $(2,2)\!:\,0.444+0=0.444$. — _Plug each point into $\lVert x-\mu\rVert^2$._

**Answer:** The minimizer is the mean $\mu=(1.333,2.0)$, and the minimum within-cluster cost is $1.111+1.111+0.444=2.667$. Any other center gives a larger sum — which is exactly why Lloyd's update step uses the mean.

</details>

**Problem 2.** One assign step can move a point between clusters; explain why that move can only lower (never raise) the distortion $J$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $J=\sum_x \lVert x-\mu_{c(x)}\rVert^2$ is a sum of independent per-point terms. — _Each point contributes only its own distance-to-its-center._
- Reassigning a point to its NEAREST center replaces its term with the smallest possible value. — _By definition of nearest, no other center gives a smaller squared distance._
- Every other point's term is unchanged (centers are held fixed during assignment). — _The assign step does not move centers._

**Answer:** Each point independently switches to the center that minimizes its own term, and nothing else changes, so the total $J$ can only stay the same or fall. Combined with the update step (which also can't raise $J$), the whole iteration is monotone non-increasing — that is the convergence guarantee. It does NOT guarantee the global optimum, only a local one.

</details>

**Problem 3.** Ablation (initialization & k): in the CODE, (a) replace k-means++ starting centers with random ones and rerun many seeds; (b) sweep $k=1\ldots5$ and look at final distortion. What do you expect, and what does the elbow tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- (a) Run Lloyd from random vs k-means++ starts over ~50 seeds; record the final $J$ of each. — _Different starts converge to different local optima._
- (b) For each $k$, run k-means and record the converged $J$. — _More centers can always fit the data more tightly._
- Plot $J$ vs $k$ and find where the curve bends. — _The 'elbow' marks diminishing returns from extra clusters._

**Answer:** (a) Random starts scatter into much worse optima (in our run, mean final $J\approx 209$, worst $\approx 547$) while k-means++ reliably hits the good optimum ($J\approx 66.23$ every time) — initialization matters a lot. (b) $J$ falls monotonically with $k$ ($\approx 1202,556,66.2,56.7,46.6$ for $k=1..5$); the big drop is at $k{=}3$, then it flattens. That elbow at $k{=}3$ matches the three blobs we generated — and it's why you can't pick $k$ by minimizing $J$ (which would push $k$ toward $n$). These are our small run, not the paper's numbers.

</details>